# Advanced Anomaly Detection Pipeline v4 — F1-Maximized
## Targeting the Recall Gap with Cross-Batch Calibration

**Diagnosis from v3 Codabench results:**
- AUC = 0.968 (excellent ranking)
- Precision = 0.831 (good)
- **Recall = 0.472 (BOTTLENECK)** — missing >50% of anomalies
- F1 = 0.602 (limited by recall)

**Root cause:** OOF threshold (~0.4) doesn't transfer to test set due to distribution shift.

**v4 strategy — 5 key changes:**
1. **Cross-batch threshold estimation** — train on 2 batches, validate on held-out 3rd batch to simulate real distribution shift
2. **Percentile-based thresholding** — "predict top X% as anomalous" instead of fixed probability cutoff (robust to score distribution shift)
3. **Unsupervised models as direct ensemble members** — IF/LOF anomaly scores in the ensemble, not just features (label-free = better generalization)
4. **Probability calibration** — isotonic regression on OOF scores for better threshold transfer
5. **Recall-biased threshold search** — use F_beta (beta=1.5) during threshold tuning to compensate for recall drop on unseen data


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
# Update this path to your working directory
os.chdir('/content/drive/My Drive/principles of machine learning/')
print('Working directory:', os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/My Drive/principles of machine learning


In [2]:
import numpy as np
import pandas as pd
import warnings, json, itertools
warnings.filterwarnings("ignore")

from scipy import stats
from scipy.sparse import csr_matrix
from scipy.stats import rankdata
from sklearn.decomposition import TruncatedSVD, NMF
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, fbeta_score, precision_score, recall_score,
                             roc_auc_score, classification_report)
from sklearn.ensemble import (IsolationForest, RandomForestClassifier,
                              GradientBoostingClassifier, ExtraTreesClassifier)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
import xgboost as xgb

print("All imports successful.")

All imports successful.


## 1. Load All Three Training Batches + Test

In [3]:
train_data_1 = np.load("training_batch_with_labels.npz")
train_data_2 = np.load("first_batch_with_labels.npz")
train_data_3 = np.load("second_batch_with_labels.npz")
test_data    = np.load("third_batch.npz")

# Keep batches separate for cross-batch validation
batches = [
    {"X": pd.DataFrame(train_data_1["X"], columns=["user","item","rating"]),
     "y": pd.DataFrame(train_data_1["y"], columns=["user","label"]).set_index("user")},
    {"X": pd.DataFrame(train_data_2["X"], columns=["user","item","rating"]),
     "y": pd.DataFrame(train_data_2["y"], columns=["user","label"]).set_index("user")},
    {"X": pd.DataFrame(train_data_3["X"], columns=["user","item","rating"]),
     "y": pd.DataFrame(train_data_3["y"], columns=["user","label"]).set_index("user")},
]

# Combined training
train_ratings = pd.concat([b["X"] for b in batches], ignore_index=True)
labels = pd.concat([b["y"] for b in batches])
test_ratings = pd.DataFrame(test_data["X"], columns=["user", "item", "rating"])
all_ratings = pd.concat([train_ratings, test_ratings], ignore_index=True)
n_items = 1000

print(f"Training: {train_ratings['user'].nunique()} users, {len(train_ratings)} interactions")
print(f"Test:     {test_ratings['user'].nunique()} users, {len(test_ratings)} interactions")
for i, b in enumerate(batches):
    n_anom = int(b['y']['label'].sum())
    print(f"  Batch {i+1}: {len(b['y'])} users, {n_anom} anomalous ({n_anom/len(b['y'])*100:.1f}%)")
print(f"\nOverall anomaly rate: {labels['label'].mean():.1%}")

Training: 3060 users, 479433 interactions
Test:     1625 users, 256072 interactions
  Batch 1: 1100 users, 100 anomalous (9.1%)
  Batch 2: 1100 users, 100 anomalous (9.1%)
  Batch 3: 860 users, 60 anomalous (7.0%)

Overall anomaly rate: 8.5%


## 2. Feature Engineering (same 127 features as v3)

In [4]:
# Precompute global item statistics from ALL data (train + test)
g_item_stats = all_ratings.groupby("item")["rating"].agg(["mean", "std", "count"])
g_item_stats.columns = ["item_mean", "item_std", "item_count"]
g_item_stats["item_std"] = g_item_stats["item_std"].fillna(0)
g_item_pop = all_ratings.groupby("item")["rating"].count()
g_mean = all_ratings["rating"].mean()
print(f"Global mean rating: {g_mean:.3f}")
print(f"Items with stats: {len(g_item_stats)}")

Global mean rating: 3.379
Items with stats: 1000


In [5]:
def compute_features(df, svd_model=None, nmf_model=None, fit=False):
    """Compute comprehensive user-level features from interaction data."""
    feats = []
    users_list = sorted(df["user"].unique())

    # ===== A. Basic Rating Statistics =====
    basic = df.groupby("user")["rating"].agg(
        mean_r="mean", std_r="std", min_r="min", max_r="max",
        med_r="median", n_ratings="count", sum_r="sum"
    ).fillna(0)
    basic["range_r"] = basic["max_r"] - basic["min_r"]
    feats.append(basic)

    # ===== B. Distribution Shape =====
    def dist_f(g):
        r = g["rating"].values; n = len(r)
        d = {}
        d["skew"] = stats.skew(r) if n >= 3 else 0
        d["kurt"] = stats.kurtosis(r) if n >= 4 else 0
        vc = pd.Series(r).value_counts(normalize=True)
        d["entropy"] = stats.entropy(vc)
        d["mode_prop"] = vc.iloc[0]
        d["mad"] = np.median(np.abs(r - np.median(r)))
        d["iqr"] = np.percentile(r, 75) - np.percentile(r, 25) if n >= 4 else 0
        d["cv"] = (np.std(r) / np.mean(r)) if np.mean(r) > 0 else 0
        for rv in range(6): d[f"pr{rv}"] = np.mean(r == rv)
        d["p_extreme"] = np.mean((r == 0) | (r == 5))
        d["p_low"] = np.mean(r <= 1)
        d["p_high"] = np.mean(r >= 4)
        d["n_distinct_r"] = len(np.unique(r))
        d["change_rate"] = np.sum(np.diff(r) != 0) / (n - 1) if n > 1 else 0
        d["concentration"] = np.sum(vc.values ** 2)
        if n > 1:
            diffs = np.diff(r)
            d["mean_abs_diff"] = np.mean(np.abs(diffs))
            d["std_diff"] = np.std(diffs)
            d["p_same_rating"] = np.mean(diffs == 0)
            d["max_run"] = max(len(list(gi)) for _, gi in itertools.groupby(r))
        else:
            d["mean_abs_diff"] = 0; d["std_diff"] = 0
            d["p_same_rating"] = 0; d["max_run"] = n
        if n >= 10:
            d["p10"] = np.percentile(r, 10)
            d["p90"] = np.percentile(r, 90)
            d["p90_p10"] = d["p90"] - d["p10"]
        else:
            d["p10"] = 0; d["p90"] = 0; d["p90_p10"] = 0
        return pd.Series(d)
    feats.append(df.groupby("user").apply(dist_f))

    # ===== C. Interaction Structure =====
    inter = df.groupby("user").agg(n_unique_items=("item", "nunique"))
    inter["repeat_ratio"] = 1 - inter["n_unique_items"] / df.groupby("user").size()
    inter["density"] = df.groupby("user").size() / n_items
    ic = df.groupby(["user", "item"]).size().reset_index(name="cnt")
    rs2 = ic.groupby("user")["cnt"].agg(
        max_item_repeat="max", mean_item_repeat="mean"
    ).fillna(0)
    inter = inter.join(rs2, how="left").fillna(0)
    feats.append(inter)

    # ===== D. Item-Normalized Deviation =====
    m = df.merge(g_item_stats, left_on="item", right_index=True, how="left")
    m["item_mean"] = m["item_mean"].fillna(g_mean)
    m["item_std"] = m["item_std"].fillna(1)
    m["resid"] = m["rating"] - m["item_mean"]
    m["z_resid"] = np.where(m["item_std"] > 0, m["resid"] / m["item_std"], 0)
    m["abs_resid"] = np.abs(m["resid"])

    r_agg = m.groupby("user").agg(
        mean_resid=("resid", "mean"), std_resid=("resid", "std"),
        abs_mean_resid=("abs_resid", "mean"), max_abs_resid=("abs_resid", "max"),
        med_resid=("resid", "median"),
        mean_z=("z_resid", "mean"), std_z=("z_resid", "std"),
        abs_mean_z=("z_resid", lambda x: np.abs(x).mean()),
        resid_q90=("abs_resid", lambda x: np.percentile(x, 90)),
        n_large_dev=("abs_resid", lambda x: (x > 2).sum()),
    ).fillna(0)
    r_agg["large_dev_ratio"] = r_agg["n_large_dev"] / df.groupby("user").size()
    n_pos = m.groupby("user")["resid"].apply(lambda x: (x > 1).sum())
    n_neg = m.groupby("user")["resid"].apply(lambda x: (x < -1).sum())
    r_agg["dev_asymmetry"] = (n_pos - n_neg) / (n_pos + n_neg + 1)
    feats.append(r_agg)

    # ===== E. Popularity-Aware =====
    mp = df.merge(g_item_pop.rename("ipop").to_frame(), left_on="item", right_index=True, how="left")
    mp["ipop"] = mp["ipop"].fillna(1)
    mp["iw"] = 1.0 / (mp["ipop"] + 1)
    mp["wr"] = mp["rating"] * mp["iw"]
    pf = mp.groupby("user").agg(
        wr_sum=("wr", "sum"), iw_sum=("iw", "sum"),
        mean_ipop=("ipop", "mean"), std_ipop=("ipop", "std"),
        mean_lp=("ipop", lambda x: np.log1p(x).mean()),
    )
    pf["w_mean_r"] = pf["wr_sum"] / pf["iw_sum"]
    pf = pf.drop(columns=["wr_sum", "iw_sum"]).fillna(0)
    mp["pop_q"] = pd.qcut(mp["ipop"].rank(method="first"), q=4, labels=False)
    for q in range(4):
        pf[f"mean_r_popq{q}"] = mp[mp["pop_q"] == q].groupby("user")["rating"].mean()
    pf = pf.fillna(0)
    feats.append(pf)

    # ===== F. Sequence Patterns =====
    def seq_f(g):
        r = g["rating"].values; n = len(r)
        d = {}
        if n >= 3:
            mid = n // 2
            d["drift"] = np.mean(r[mid:]) - np.mean(r[:mid])
            d["trend"] = np.polyfit(range(n), r, 1)[0]
            d["autocorr"] = np.corrcoef(r[:-1], r[1:])[0, 1] if np.std(r) > 0 else 0
            if np.isnan(d["autocorr"]): d["autocorr"] = 0
        else:
            d["drift"] = 0; d["trend"] = 0; d["autocorr"] = 0
        if n >= 10:
            q1, q2, q3, q4 = np.array_split(r, 4)
            d["q4_q1_diff"] = np.mean(q4) - np.mean(q1)
            d["segment_std"] = np.std([np.mean(q1), np.mean(q2), np.mean(q3), np.mean(q4)])
        else:
            d["q4_q1_diff"] = 0; d["segment_std"] = 0
        return pd.Series(d)
    feats.append(df.groupby("user").apply(seq_f))

    # ===== G. Item Diversity =====
    def div_f(g):
        vc = pd.Series(g["item"].values).value_counts(normalize=True)
        return pd.Series({
            "item_entropy": stats.entropy(vc),
            "gini_simpson": 1 - np.sum(vc ** 2),
            "unique_ratio": len(np.unique(g["item"].values)) / n_items,
        })
    feats.append(df.groupby("user").apply(div_f))

    # ===== H. SVD Latent Embeddings (35 components) =====
    uid_map = {u: i for i, u in enumerate(users_list)}
    rows = df["user"].map(uid_map).values
    cols = df["item"].values
    vals = df["rating"].values.astype(float)
    um = csr_matrix((vals, (rows, cols)), shape=(len(users_list), n_items))
    nc_svd = 35
    if fit or svd_model is None:
        svd_model = TruncatedSVD(n_components=nc_svd, random_state=42)
        emb = svd_model.fit_transform(um)
    else:
        emb = svd_model.transform(um)
    svd_df = pd.DataFrame(emb, index=users_list, columns=[f"svd_{i}" for i in range(nc_svd)])
    svd_df.index.name = "user"
    recon = svd_model.inverse_transform(emb)
    svd_df["svd_err"] = np.mean((um.toarray() - recon) ** 2, axis=1)
    svd_df["svd_norm"] = np.linalg.norm(emb, axis=1)
    feats.append(svd_df)

    # ===== H2. NMF Embeddings (15 components) =====
    nc_nmf = 15
    um_pos = um.copy()
    um_pos.data = np.clip(um_pos.data, 0, None)
    if fit or nmf_model is None:
        nmf_model = NMF(n_components=nc_nmf, random_state=42, max_iter=300, init='nndsvda')
        nmf_emb = nmf_model.fit_transform(um_pos)
    else:
        nmf_emb = nmf_model.transform(um_pos)
    nmf_df = pd.DataFrame(nmf_emb, index=users_list, columns=[f"nmf_{i}" for i in range(nc_nmf)])
    nmf_df.index.name = "user"
    nmf_recon = nmf_model.inverse_transform(nmf_emb)
    nmf_df["nmf_err"] = np.mean((um_pos.toarray() - nmf_recon) ** 2, axis=1)
    nmf_df["nmf_norm"] = np.linalg.norm(nmf_emb, axis=1)
    feats.append(nmf_df)

    # ===== I. Cosine Similarity to Average User =====
    bm = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(users_list), n_items))
    avg = bm.mean(axis=0).A1
    sims = []
    for i in range(len(users_list)):
        uv = bm[i].toarray().flatten()
        d_val = np.dot(uv, avg); nu = np.linalg.norm(uv); na = np.linalg.norm(avg)
        sims.append(d_val / (nu * na) if nu > 0 and na > 0 else 0)
    feats.append(pd.DataFrame({"cos_sim_avg": sims}, index=users_list).rename_axis("user"))

    # ===== J. Rating vs Item Popularity Correlation =====
    mp2 = df.merge(g_item_pop.rename("ipop").to_frame(), left_on="item", right_index=True, how="left")
    mp2["ipop"] = mp2["ipop"].fillna(1)
    def pcorr(g):
        if len(g) < 5: return pd.Series({"rating_pop_corr": 0})
        c = np.corrcoef(g["rating"].values, g["ipop"].values)[0, 1]
        return pd.Series({"rating_pop_corr": c if not np.isnan(c) else 0})
    feats.append(mp2.groupby("user").apply(pcorr))

    # ===== K. Item Overlap Between Halves =====
    def burst_f(g):
        items = g["item"].values; n = len(items)
        if n >= 10:
            h1 = set(items[:n//2]); h2 = set(items[n//2:])
            return pd.Series({"item_overlap_halves": len(h1 & h2) / max(len(h1 | h2), 1)})
        return pd.Series({"item_overlap_halves": 0})
    feats.append(df.groupby("user").apply(burst_f))

    result = feats[0]
    for f in feats[1:]: result = result.join(f, how="left")
    result = result.fillna(0).replace([np.inf, -np.inf], 0)
    return result, svd_model, nmf_model

print("Computing training features...")
X_train_f, svd_m, nmf_m = compute_features(train_ratings, fit=True)
print("Computing test features...")
X_test_f, _, _ = compute_features(test_ratings, svd_model=svd_m, nmf_model=nmf_m, fit=False)
y_train = labels.loc[X_train_f.index, "label"]

print(f"\nTraining features: {X_train_f.shape}")
print(f"Test features:     {X_test_f.shape}")

Computing training features...
Computing test features...

Training features: (3060, 124)
Test features:     (1625, 124)


## 3. Unsupervised Anomaly Scores (as features AND as ensemble members)

In [6]:
sc_u = RobustScaler()
Xts = sc_u.fit_transform(X_train_f)
Xes = sc_u.transform(X_test_f)

# --- Unsupervised scores (used both as features AND as standalone ensemble members) ---
unsup_train_scores = {}  # store for ensemble
unsup_test_scores = {}

# Multiple Isolation Forest configs
for i, (cont, mf, rs) in enumerate([(0.085, 0.8, 42), (0.095, 0.6, 77)]):
    iso = IsolationForest(n_estimators=300, contamination=cont, random_state=rs, max_features=mf)
    iso.fit(Xts)
    tr_score = -iso.score_samples(Xts)
    te_score = -iso.score_samples(Xes)
    X_train_f[f"iso_score_{i}"] = tr_score
    X_test_f[f"iso_score_{i}"] = te_score
    unsup_train_scores[f"iso_{i}"] = tr_score
    unsup_test_scores[f"iso_{i}"] = te_score

# LOF
lof = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.09)
lof.fit(Xts)
tr_score = -lof.score_samples(Xts)
te_score = -lof.score_samples(Xes)
X_train_f["lof_score"] = tr_score
X_test_f["lof_score"] = te_score
unsup_train_scores["lof"] = tr_score
unsup_test_scores["lof"] = te_score

# LOF with different k
lof2 = LocalOutlierFactor(n_neighbors=50, novelty=True, contamination=0.09)
lof2.fit(Xts)
unsup_train_scores["lof_k50"] = -lof2.score_samples(Xts)
unsup_test_scores["lof_k50"] = -lof2.score_samples(Xes)

print(f"Final feature count: {X_train_f.shape[1]}")
print(f"Unsupervised ensemble members: {list(unsup_train_scores.keys())}")

Final feature count: 127
Unsupervised ensemble members: ['iso_0', 'iso_1', 'lof', 'lof_k50']


## 3b. [NEW] Cross-Batch Validation for Threshold Estimation

**This is the key innovation in v4.** Instead of relying on within-batch OOF thresholds
(which don't transfer to unseen test data), we simulate the real evaluation scenario:
- Train on 2 batches, predict on the 3rd held-out batch
- Find optimal threshold on the held-out batch
- Average these cross-batch thresholds for a robust estimate

This captures the distribution shift between training and test batches.

In [7]:
# --- Cross-batch validation to estimate realistic threshold ---
print("Cross-Batch Validation for Threshold Estimation")
print("=" * 60)

cross_batch_thresholds = []
cross_batch_f1s = []
cross_batch_pctiles = []  # track what percentile the threshold corresponds to

for hold_idx in range(3):
    # Train batches = everything except hold_idx
    train_b = [b for i, b in enumerate(batches) if i != hold_idx]
    val_b = batches[hold_idx]

    cb_train_ratings = pd.concat([b["X"] for b in train_b], ignore_index=True)
    cb_labels = pd.concat([b["y"] for b in train_b])
    cb_val_ratings = val_b["X"]
    cb_val_labels = val_b["y"]

    # Compute features (reuse global item stats for consistency)
    cb_train_f, cb_svd, cb_nmf = compute_features(cb_train_ratings, fit=True)
    cb_val_f, _, _ = compute_features(cb_val_ratings, svd_model=cb_svd, nmf_model=cb_nmf, fit=False)
    cb_y_train = cb_labels.loc[cb_train_f.index, "label"].values
    cb_y_val = cb_val_labels.loc[cb_val_f.index, "label"].values

    # Add unsupervised features
    cb_sc = RobustScaler()
    cb_Xts = cb_sc.fit_transform(cb_train_f)
    cb_Xes = cb_sc.transform(cb_val_f)
    for i, (cont, mf, rs) in enumerate([(0.085, 0.8, 42), (0.095, 0.6, 77)]):
        iso_cb = IsolationForest(n_estimators=300, contamination=cont, random_state=rs, max_features=mf)
        iso_cb.fit(cb_Xts)
        cb_train_f[f"iso_score_{i}"] = -iso_cb.score_samples(cb_Xts)
        cb_val_f[f"iso_score_{i}"] = -iso_cb.score_samples(cb_Xes)
    lof_cb = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.09)
    lof_cb.fit(cb_Xts)
    cb_train_f["lof_score"] = -lof_cb.score_samples(cb_Xts)
    cb_val_f["lof_score"] = -lof_cb.score_samples(cb_Xes)

    cb_X = cb_train_f.values
    cb_Xval = cb_val_f.values
    cb_spw = (len(cb_y_train) - cb_y_train.sum()) / cb_y_train.sum()

    # Train a quick LightGBM ensemble on training batches
    cb_preds = np.zeros(len(cb_y_val))
    n_models_cb = 0
    for seed in [42, 77, 123]:
        m = lgb.LGBMClassifier(
            n_estimators=500, learning_rate=0.025, max_depth=5, num_leaves=20,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.65,
            reg_alpha=0.8, reg_lambda=1.5, scale_pos_weight=cb_spw,
            random_state=seed, verbose=-1)
        m.fit(cb_X, cb_y_train)
        cb_preds += m.predict_proba(cb_Xval)[:, 1]
        n_models_cb += 1
    cb_preds /= n_models_cb

    # Find optimal threshold on held-out batch
    best_f1_cb, best_thr_cb = 0, 0.5
    for t in np.arange(0.01, 0.99, 0.005):
        f = f1_score(cb_y_val, (cb_preds >= t).astype(int))
        if f > best_f1_cb: best_f1_cb, best_thr_cb = f, t

    # What percentile does this threshold correspond to in the score distribution?
    pctile = np.mean(cb_preds < best_thr_cb) * 100
    n_pred = (cb_preds >= best_thr_cb).sum()

    cross_batch_thresholds.append(best_thr_cb)
    cross_batch_f1s.append(best_f1_cb)
    cross_batch_pctiles.append(pctile)

    print(f"  Hold-out batch {hold_idx+1}: thr={best_thr_cb:.3f}, F1={best_f1_cb:.3f}, "
          f"pred_anom={n_pred}/{len(cb_y_val)} ({n_pred/len(cb_y_val)*100:.1f}%), "
          f"score_pctile={pctile:.1f}%")

# Cross-batch threshold estimate
cb_threshold = np.median(cross_batch_thresholds)
cb_pctile = np.median(cross_batch_pctiles)
print(f"\nCross-batch threshold estimate:")
print(f"  Median threshold: {cb_threshold:.3f}")
print(f"  Median score percentile: {cb_pctile:.1f}%")
print(f"  Mean F1 on held-out batches: {np.mean(cross_batch_f1s):.3f}")

Cross-Batch Validation for Threshold Estimation
  Hold-out batch 1: thr=0.480, F1=0.862, pred_anom=81/1100 (7.4%), score_pctile=92.6%
  Hold-out batch 2: thr=0.480, F1=0.841, pred_anom=95/1100 (8.6%), score_pctile=91.4%
  Hold-out batch 3: thr=0.270, F1=0.514, pred_anom=45/860 (5.2%), score_pctile=94.8%

Cross-batch threshold estimate:
  Median threshold: 0.480
  Median score percentile: 92.6%
  Mean F1 on held-out batches: 0.739


## 4. Model Training with Multi-Seed 10-Fold CV

Same model set as v3 but with **isotonic calibration** of OOF scores
and unsupervised models included as ensemble members.

In [8]:
fcols = X_train_f.columns.tolist()
X = X_train_f[fcols].values
Xt = X_test_f[fcols].values
y = y_train.values

sc2 = RobustScaler()
Xs = sc2.fit_transform(X)
Xts2 = sc2.transform(Xt)

spw = (len(y) - y.sum()) / y.sum()
print(f"Class imbalance: {spw:.0f}:1 (normal:anomalous)")

def get_models(seed=0):
    return {
        f"lgbm_a_{seed}": lgb.LGBMClassifier(
            n_estimators=500, learning_rate=0.025, max_depth=5, num_leaves=20,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.65,
            reg_alpha=0.8, reg_lambda=1.5, scale_pos_weight=spw,
            random_state=42+seed, verbose=-1),
        f"lgbm_b_{seed}": lgb.LGBMClassifier(
            n_estimators=400, learning_rate=0.035, max_depth=4, num_leaves=12,
            min_child_samples=15, subsample=0.75, colsample_bytree=0.55,
            reg_alpha=1.5, reg_lambda=2.5, scale_pos_weight=spw,
            random_state=77+seed, verbose=-1),
        f"lgbm_c_{seed}": lgb.LGBMClassifier(
            n_estimators=450, learning_rate=0.03, max_depth=6, num_leaves=25,
            min_child_samples=8, subsample=0.85, colsample_bytree=0.7,
            reg_alpha=0.5, reg_lambda=1.0, scale_pos_weight=spw,
            random_state=123+seed, verbose=-1),
        f"lgbm_d_{seed}": lgb.LGBMClassifier(
            n_estimators=350, learning_rate=0.04, max_depth=3, num_leaves=8,
            min_child_samples=20, subsample=0.7, colsample_bytree=0.5,
            reg_alpha=2.0, reg_lambda=3.0, scale_pos_weight=spw,
            random_state=200+seed, verbose=-1),
        f"xgb_a_{seed}": xgb.XGBClassifier(
            n_estimators=500, learning_rate=0.025, max_depth=5,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.65,
            reg_alpha=0.8, reg_lambda=1.5, scale_pos_weight=spw,
            random_state=42+seed, eval_metric="logloss", verbosity=0),
        f"xgb_b_{seed}": xgb.XGBClassifier(
            n_estimators=400, learning_rate=0.035, max_depth=4,
            min_child_weight=8, subsample=0.75, colsample_bytree=0.55,
            reg_alpha=1.5, reg_lambda=2.5, scale_pos_weight=spw,
            random_state=77+seed, eval_metric="logloss", verbosity=0),
        f"xgb_c_{seed}": xgb.XGBClassifier(
            n_estimators=450, learning_rate=0.03, max_depth=6,
            min_child_weight=3, subsample=0.85, colsample_bytree=0.7,
            reg_alpha=0.5, reg_lambda=1.0, scale_pos_weight=spw,
            random_state=123+seed, eval_metric="logloss", verbosity=0),
        f"rf_{seed}": RandomForestClassifier(
            n_estimators=500, max_depth=8, min_samples_leaf=4,
            class_weight="balanced", random_state=42+seed, n_jobs=-1),
        f"et_{seed}": ExtraTreesClassifier(
            n_estimators=500, max_depth=8, min_samples_leaf=4,
            class_weight="balanced", random_state=42+seed, n_jobs=-1),
        f"gb_{seed}": GradientBoostingClassifier(
            n_estimators=350, learning_rate=0.035, max_depth=4,
            min_samples_leaf=8, subsample=0.8, random_state=42+seed),
        f"lr_{seed}": LogisticRegression(
            C=0.5, class_weight="balanced", max_iter=3000, random_state=42+seed),
    }

print(f"Models per seed: {len(get_models(0))}")
print(f"Total model variants: {len(get_models(0))} × 2 seeds = {len(get_models(0))*2}")

Class imbalance: 11:1 (normal:anomalous)
Models per seed: 11
Total model variants: 11 × 2 seeds = 22


In [9]:
N_FOLDS = 10
all_oof = {}
all_test_p = {}

for seed in [0, 1000]:
    models_dict = get_models(seed)
    model_names = list(models_dict.keys())
    oof = {n: np.zeros(len(y)) for n in model_names}
    test_p = {n: np.zeros(len(Xt)) for n in model_names}

    cv_seed = 42 + seed
    print(f"\n{'='*60}")
    print(f"Seed={seed}: {N_FOLDS}-fold stratified CV ({len(model_names)} models)")
    print(f"{'='*60}")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=cv_seed)

    for fi, (tidx, vidx) in enumerate(skf.split(X, y)):
        Xtr, Xv = X[tidx], X[vidx]
        Xtr_s, Xv_s = Xs[tidx], Xs[vidx]
        ytr, yv = y[tidx], y[vidx]

        fold_models = get_models(seed)
        fold_info = []
        for name in model_names:
            model = fold_models[name]
            if "lr" in name:
                model.fit(Xtr_s, ytr)
                oof[name][vidx] = model.predict_proba(Xv_s)[:, 1]
                test_p[name] += model.predict_proba(Xts2)[:, 1] / N_FOLDS
            else:
                model.fit(Xtr, ytr)
                oof[name][vidx] = model.predict_proba(Xv)[:, 1]
                test_p[name] += model.predict_proba(Xt)[:, 1] / N_FOLDS

            bf = max(f1_score(yv, (oof[name][vidx] >= t).astype(int))
                     for t in np.arange(0.05, 0.95, 0.01))
            fold_info.append(f"{name.split('_' + str(seed))[0]}:{bf:.3f}")

        print(f"  Fold {fi+1}: {', '.join(fold_info)}")

    all_oof.update(oof)
    all_test_p.update(test_p)

    print(f"\nSeed {seed} OOF Performance:")
    for n in model_names:
        auc = roc_auc_score(y, oof[n])
        bf = max(f1_score(y, (oof[n] >= t).astype(int)) for t in np.arange(0.01, 0.99, 0.005))
        print(f"  {n:17s}: AUC={auc:.4f}, F1={bf:.4f}")

# --- Add unsupervised models as ensemble members ---
# Normalize unsup scores to [0,1] range using rank-percentile
for name in unsup_train_scores:
    all_oof[f"unsup_{name}"] = rankdata(unsup_train_scores[name]) / len(y)
    all_test_p[f"unsup_{name}"] = rankdata(unsup_test_scores[name]) / len(Xt)

print(f"\nTotal models (supervised + unsupervised): {len(all_oof)}")


Seed=0: 10-fold stratified CV (11 models)
  Fold 1: lgbm_a:0.756, lgbm_b:0.776, lgbm_c:0.756, lgbm_d:0.756, xgb_a:0.766, xgb_b:0.744, xgb_c:0.783, rf:0.723, et:0.714, gb:0.727, lr:0.708
  Fold 2: lgbm_a:0.808, lgbm_b:0.800, lgbm_c:0.800, lgbm_d:0.776, xgb_a:0.846, xgb_b:0.824, xgb_c:0.830, rf:0.760, et:0.645, gb:0.840, lr:0.792
  Fold 3: lgbm_a:0.902, lgbm_b:0.902, lgbm_c:0.885, lgbm_d:0.873, xgb_a:0.909, xgb_b:0.893, xgb_c:0.909, rf:0.857, et:0.741, gb:0.863, lr:0.863
  Fold 4: lgbm_a:0.791, lgbm_b:0.780, lgbm_c:0.808, lgbm_d:0.784, xgb_a:0.800, xgb_b:0.816, xgb_c:0.816, rf:0.756, et:0.735, gb:0.833, lr:0.735
  Fold 5: lgbm_a:0.880, lgbm_b:0.880, lgbm_c:0.852, lgbm_d:0.863, xgb_a:0.880, xgb_b:0.863, xgb_c:0.885, rf:0.792, et:0.688, gb:0.824, lr:0.689
  Fold 6: lgbm_a:0.875, lgbm_b:0.898, lgbm_c:0.875, lgbm_d:0.863, xgb_a:0.863, xgb_b:0.880, xgb_c:0.857, rf:0.760, et:0.654, gb:0.840, lr:0.727
  Fold 7: lgbm_a:0.880, lgbm_b:0.857, lgbm_c:0.846, lgbm_d:0.868, xgb_a:0.863, xgb_b:0.852, x

## 5. [NEW] Isotonic Calibration of OOF Scores

Isotonic regression maps OOF scores → calibrated probabilities.
This makes thresholds more transferable between OOF and test data.

In [10]:
# Calibrate each model's scores using isotonic regression
all_oof_cal = {}
all_test_cal = {}

for name in all_oof:
    if name.startswith("unsup_"):
        # Unsupervised: just pass through (already rank-normalized)
        all_oof_cal[name] = all_oof[name]
        all_test_cal[name] = all_test_p[name]
    else:
        # Supervised: apply isotonic calibration
        iso_reg = IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=1)
        iso_reg.fit(all_oof[name], y)
        all_oof_cal[name] = iso_reg.predict(all_oof[name])
        all_test_cal[name] = iso_reg.predict(all_test_p[name])

print("Calibrated all model scores.")
# Check calibration effect on a sample model
sample = list(all_oof.keys())[0]
print(f"\nCalibration effect on '{sample}':")
print(f"  Raw  - mean: {all_oof[sample].mean():.4f}, std: {all_oof[sample].std():.4f}")
print(f"  Calib - mean: {all_oof_cal[sample].mean():.4f}, std: {all_oof_cal[sample].std():.4f}")

Calibrated all model scores.

Calibration effect on 'lgbm_a_0':
  Raw  - mean: 0.0888, std: 0.2384
  Calib - mean: 0.0850, std: 0.2339


## 6. Ensemble Selection with Multiple Threshold Strategies

**Three threshold strategies compared:**
1. **Standard F1** — maximize F1 on OOF (what v3 did)
2. **Cross-batch threshold** — from Section 3b cross-batch validation
3. **Percentile-based** — predict top X% as anomalous (where X matches training anomaly rate)
4. **F_beta biased** — use beta=1.5 to favor recall during threshold selection

In [11]:
def best_f1_thr(scores, y_true):
    bf, bt = 0, 0.5
    for t in np.arange(0.01, 0.99, 0.005):
        f = f1_score(y_true, (scores >= t).astype(int))
        if f > bf: bf, bt = f, t
    return bf, bt

def best_fbeta_thr(scores, y_true, beta=1.5):
    bf, bt = 0, 0.5
    for t in np.arange(0.01, 0.99, 0.005):
        f = fbeta_score(y_true, (scores >= t).astype(int), beta=beta)
        if f > bf: bf, bt = f, t
    return bf, bt

def percentile_thr(scores, target_pct):
    """Threshold such that top target_pct% of scores are predicted positive."""
    return np.percentile(scores, 100 - target_pct)

all_model_names = list(all_oof_cal.keys())
model_aucs = {}
for n in all_model_names:
    try:
        model_aucs[n] = roc_auc_score(y, all_oof_cal[n])
    except:
        model_aucs[n] = 0.5
sorted_m = sorted(model_aucs, key=model_aucs.get, reverse=True)

# Only use models with AUC > 0.75
good_models = [n for n in sorted_m if model_aucs[n] > 0.75]
print(f"Models with AUC > 0.75: {len(good_models)} / {len(sorted_m)}")

results = {}
train_anomaly_rate = y.mean() * 100  # ~8.5%

# ======= Build ensemble candidates =======
ensemble_scores = {}

# Individual top models
for n in good_models[:10]:
    ensemble_scores[f"solo_{n[:14]}"] = (all_oof_cal[n], all_test_cal[n])

# Multi-seed averaging
base_names = set()
for n in all_model_names:
    for sfx in ["_0", "_1000"]:
        if n.endswith(sfx): base_names.add(n[:-len(sfx)])
for bn in sorted(base_names):
    matching = [n for n in all_model_names if any(n == f"{bn}_{s}" for s in [0, 1000])]
    if len(matching) >= 2:
        avg_oof = np.mean([all_oof_cal[n] for n in matching], axis=0)
        avg_test = np.mean([all_test_cal[n] for n in matching], axis=0)
        ensemble_scores[f"mseed_{bn}"] = (avg_oof, avg_test)

# Top-N averaging (calibrated scores)
for top_n in [5, 8, 10, 15, 20]:
    top = good_models[:top_n]
    oof_e = np.mean([all_oof_cal[n] for n in top], axis=0)
    test_e = np.mean([all_test_cal[n] for n in top], axis=0)
    ensemble_scores[f"top{top_n}_eq_cal"] = (oof_e, test_e)

# Top-N rank averaging (most robust to score scale)
for top_n in [5, 8, 10, 15, 20]:
    top = good_models[:top_n]
    oof_e = np.mean([rankdata(all_oof_cal[n])/len(y) for n in top], axis=0)
    test_e = np.mean([rankdata(all_test_cal[n])/len(Xt) for n in top], axis=0)
    ensemble_scores[f"top{top_n}_rank"] = (oof_e, test_e)

# Stacking
for top_n in [8, 10, 15]:
    top = good_models[:top_n]
    s_X = np.column_stack([all_oof_cal[n] for n in top])
    s_Xt = np.column_stack([all_test_cal[n] for n in top])
    s_X = np.hstack([s_X, np.column_stack([rankdata(all_oof_cal[n])/len(y) for n in top])])
    s_Xt = np.hstack([s_Xt, np.column_stack([rankdata(all_test_cal[n])/len(Xt) for n in top])])

    m_oof = np.zeros(len(y)); m_test = np.zeros(len(Xt))
    n_C = 3
    skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for ti, vi in skf2.split(s_X, y):
        for C in [0.1, 0.5, 1.0]:
            mm = LogisticRegression(C=C, class_weight="balanced", max_iter=3000, random_state=42)
            mm.fit(s_X[ti], y[ti])
            m_oof[vi] += mm.predict_proba(s_X[vi])[:, 1] / n_C
            m_test += mm.predict_proba(s_Xt)[:, 1] / (5 * n_C)
    ensemble_scores[f"stack_top{top_n}"] = (m_oof, m_test)

print(f"\nTotal ensemble candidates: {len(ensemble_scores)}")

Models with AUC > 0.75: 22 / 26

Total ensemble candidates: 34


In [12]:
# ======= Evaluate each ensemble × threshold strategy =======
print("Evaluating all ensemble × threshold combinations...")
print("=" * 80)

all_results = []  # (name, strategy, oof_f1, thr, oof_scores, test_scores)

for ens_name, (oof_s, test_s) in ensemble_scores.items():
    # Strategy 1: Standard F1 threshold on OOF
    f1_std, thr_std = best_f1_thr(oof_s, y)
    all_results.append((ens_name, "std_f1", f1_std, thr_std, oof_s, test_s))

    # Strategy 2: F_beta=1.5 threshold (recall-biased)
    _, thr_fb = best_fbeta_thr(oof_s, y, beta=1.5)
    f1_fb = f1_score(y, (oof_s >= thr_fb).astype(int))
    all_results.append((ens_name, "fbeta1.5", f1_fb, thr_fb, oof_s, test_s))

    # Strategy 3: Percentile-based (match training anomaly rate)
    # Try a range: 8%, 9%, 10%, 11%, 12% to find best
    for pct in [8, 9, 10, 11, 12, 14]:
        thr_pct = percentile_thr(oof_s, pct)
        f1_pct = f1_score(y, (oof_s >= thr_pct).astype(int))
        all_results.append((ens_name, f"pctile_{pct}", f1_pct, thr_pct, oof_s, test_s))

    # Strategy 4: Cross-batch threshold (applied to calibrated scores)
    # Use the percentile from cross-batch validation
    thr_cb = percentile_thr(oof_s, 100 - cb_pctile)
    f1_cb = f1_score(y, (oof_s >= thr_cb).astype(int))
    all_results.append((ens_name, "cross_batch", f1_cb, thr_cb, oof_s, test_s))

# Sort by OOF F1
all_results.sort(key=lambda x: -x[2])

print(f"\nTop 30 configurations (ensemble × threshold):")
print(f"{'Rank':>4}  {'Ensemble':22s}  {'Strategy':12s}  {'OOF_F1':>7}  {'Thr':>6}  {'Pred%':>5}")
print("-" * 70)
for i, (ens, strat, f1, thr, oof_s, test_s) in enumerate(all_results[:30]):
    pred_pct = (oof_s >= thr).mean() * 100
    marker = " <<<" if i == 0 else ""
    print(f"  {i+1:2d}. {ens:22s}  {strat:12s}  {f1:.4f}  {thr:.3f}  {pred_pct:5.1f}%{marker}")

Evaluating all ensemble × threshold combinations...

Top 30 configurations (ensemble × threshold):
Rank  Ensemble                Strategy       OOF_F1     Thr  Pred%
----------------------------------------------------------------------
   1. mseed_xgb_c             std_f1        0.8391  0.395    8.6% <<<
   2. top20_rank              std_f1        0.8381  0.905    8.7%
   3. top8_eq_cal             std_f1        0.8372  0.415    8.4%
   4. top20_eq_cal            std_f1        0.8363  0.415    8.1%
   5. top20_rank              fbeta1.5      0.8362  0.900    8.9%
   6. top20_eq_cal            pctile_8      0.8356  0.425    8.0%
   7. top20_rank              pctile_8      0.8356  0.912    8.0%
   8. top10_eq_cal            std_f1        0.8352  0.380    8.6%
   9. top5_eq_cal             std_f1        0.8343  0.420    8.3%
  10. mseed_xgb_c             pctile_8      0.8337  0.484    8.2%
  11. stack_top8              std_f1        0.8333  0.880    8.4%
  12. mseed_lgbm_b            std

## 7. Smart Selection: Balance OOF F1 vs Generalization

**Key insight:** The v3 failure was picking the config with highest OOF F1,
which had too-high threshold → low recall on test. Instead, we:
1. Take top-10 configs by OOF F1
2. Among those, prefer configs that predict **~9-12% anomalous** (matching expected rate)
3. Prefer recall-biased or percentile strategies over raw F1-optimal threshold
4. Use the cross-batch threshold as a sanity check

In [13]:
# ======= Final Selection Strategy =======
# We want the config that maximizes TEST F1, not just OOF F1.
# From v3 results: OOF F1=0.835 but test F1=0.602 (precision=0.83, recall=0.47).
# This means threshold was too high → predicted too few anomalies (~4.5%).
# True anomaly rate is ~8.5%. We should predict 8-12% as anomalous.

print("=" * 60)
print("FINAL SELECTION WITH GENERALIZATION AWARENESS")
print("=" * 60)

# For each ensemble, compare strategies and pick the one most likely to generalize
# We'll produce MULTIPLE submission files so you can try several on Codabench

submissions = []

# --- Submission A: Best OOF F1 (baseline, like v3) ---
best = all_results[0]
submissions.append(("A_best_oof_f1", best[4], best[5], best[3], best[2]))

# --- Submission B: Best ensemble + percentile threshold targeting 10% anomaly rate ---
# Find best ensemble by OOF AUC, then apply 10% threshold
# Get unique ensemble names sorted by best OOF F1
seen_ens = set()
best_per_ens = []
for (ens, strat, f1, thr, oof_s, test_s) in all_results:
    if ens not in seen_ens:
        seen_ens.add(ens)
        best_per_ens.append((ens, f1, oof_s, test_s))

# Take top ensemble, apply percentile thresholds
top_ens_name, top_ens_f1, top_ens_oof, top_ens_test = best_per_ens[0]
for pct in [10, 11, 12]:
    thr_pct = percentile_thr(top_ens_test, pct)  # Apply to TEST scores
    oof_thr = percentile_thr(top_ens_oof, pct)
    oof_f1 = f1_score(y, (top_ens_oof >= oof_thr).astype(int))
    submissions.append((f"B_pctile{pct}_{top_ens_name[:15]}", top_ens_oof, top_ens_test, thr_pct, oof_f1))

# --- Submission C: Best F_beta=1.5 config ---
fbeta_results = [(ens, strat, f1, thr, oof_s, test_s) for (ens, strat, f1, thr, oof_s, test_s) in all_results
                 if strat == "fbeta1.5"]
fbeta_results.sort(key=lambda x: -x[2])
if fbeta_results:
    fb = fbeta_results[0]
    submissions.append(("C_fbeta_recall_biased", fb[4], fb[5], fb[3], fb[2]))

# --- Submission D: Cross-batch threshold with best ensemble ---
cb_thr_test = percentile_thr(top_ens_test, 100 - cb_pctile)
cb_oof_thr = percentile_thr(top_ens_oof, 100 - cb_pctile)
cb_oof_f1 = f1_score(y, (top_ens_oof >= cb_oof_thr).astype(int))
submissions.append(("D_cross_batch_thr", top_ens_oof, top_ens_test, cb_thr_test, cb_oof_f1))

# --- Submission E: Rank-based ensemble + percentile ---
rank_results = [(ens, strat, f1, thr, oof_s, test_s) for (ens, strat, f1, thr, oof_s, test_s) in all_results
                if "rank" in ens and strat.startswith("pctile_1")]
rank_results.sort(key=lambda x: -x[2])
if rank_results:
    rk = rank_results[0]
    submissions.append((f"E_rank_{rk[1]}", rk[4], rk[5], rk[3], rk[2]))

print(f"\nGenerated {len(submissions)} submission variants:")
print(f"{'Name':40s} {'OOF_F1':>7} {'Thr':>6} {'Pred%(oof)':>9} {'Pred%(test)':>10}")
print("-" * 75)
for name, oof_s, test_s, thr, oof_f1 in submissions:
    oof_pct = (oof_s >= thr).mean() * 100 if not np.isnan(thr) else 0
    test_pct = (test_s >= thr).mean() * 100 if not np.isnan(thr) else 0
    print(f"  {name:40s} {oof_f1:.4f} {thr:.3f}   {oof_pct:5.1f}%      {test_pct:5.1f}%")

FINAL SELECTION WITH GENERALIZATION AWARENESS

Generated 7 submission variants:
Name                                      OOF_F1    Thr Pred%(oof) Pred%(test)
---------------------------------------------------------------------------
  A_best_oof_f1                            0.8391 0.395     8.6%        6.1%
  B_pctile10_mseed_xgb_c                   0.8127 0.113    12.3%       11.4%
  B_pctile11_mseed_xgb_c                   0.7873 0.113    12.3%       11.4%
  B_pctile12_mseed_xgb_c                   0.7547 0.083    13.6%       13.9%
  C_fbeta_recall_biased                    0.8362 0.900     8.9%        8.2%
  D_cross_batch_thr                        0.8148 0.335     8.9%        7.5%
  E_rank_pctile_10                         0.8092 0.884    10.0%        9.7%


## 8. Save All Submission Variants

In [14]:
test_users = X_test_f.index.values

for sub_name, oof_s, test_s, thr, oof_f1 in submissions:
    pred_dict = {int(u): float(s) for u, s in zip(test_users, test_s)}

    # Save .npz
    np.savez(f"submission_{sub_name}.npz", predictions=test_s)

    # Save .json
    with open(f"predictions_{sub_name}.json", "w") as f:
        json.dump({"predictions": pred_dict}, f)

    # Save .csv
    res_df = pd.DataFrame({
        "user": test_users, "anomaly_score": test_s,
        "predicted_label": (test_s >= thr).astype(int)
    }).sort_values("anomaly_score", ascending=False)
    res_df.to_csv(f"predictions_{sub_name}.csv", index=False)

    n_anom = (test_s >= thr).sum()
    print(f"\n{sub_name}:")
    print(f"  Predicted anomalies: {n_anom}/{len(test_users)} ({n_anom/len(test_users)*100:.1f}%)")
    print(f"  Score: mean={test_s.mean():.4f}, std={test_s.std():.4f}")
    print(f"  Top 10 most anomalous: {res_df.head(10)['user'].tolist()}")


A_best_oof_f1:
  Predicted anomalies: 99/1625 (6.1%)
  Score: mean=0.0628, std=0.1764
  Top 10 most anomalous: [6384, 5643, 5565, 6170, 6535, 5635, 7005, 5905, 6207, 6088]

B_pctile10_mseed_xgb_c:
  Predicted anomalies: 186/1625 (11.4%)
  Score: mean=0.0628, std=0.1764
  Top 10 most anomalous: [6384, 5643, 5565, 6170, 6535, 5635, 7005, 5905, 6207, 6088]

B_pctile11_mseed_xgb_c:
  Predicted anomalies: 186/1625 (11.4%)
  Score: mean=0.0628, std=0.1764
  Top 10 most anomalous: [6384, 5643, 5565, 6170, 6535, 5635, 7005, 5905, 6207, 6088]

B_pctile12_mseed_xgb_c:
  Predicted anomalies: 226/1625 (13.9%)
  Score: mean=0.0628, std=0.1764
  Top 10 most anomalous: [6384, 5643, 5565, 6170, 6535, 5635, 7005, 5905, 6207, 6088]

C_fbeta_recall_biased:
  Predicted anomalies: 133/1625 (8.2%)
  Score: mean=0.5003, std=0.2536
  Top 10 most anomalous: [5643, 6384, 5565, 6170, 7102, 6535, 6881, 6036, 6207, 5635]

D_cross_batch_thr:
  Predicted anomalies: 122/1625 (7.5%)
  Score: mean=0.0628, std=0.1764
 

## 9. Primary Submission (Recommended)

**The recommended submission uses percentile-based thresholding at ~10-11%**,
which should predict a realistic number of anomalies and avoid the recall collapse
we saw in v3. If the test anomaly rate is similar to training (~8.5%),
predicting slightly above that compensates for imperfect ranking.

In [15]:
# === PRIMARY SUBMISSION ===
# Use the best-AUC ensemble with percentile threshold targeting 11% anomaly prediction
# This is the sweet spot: slightly above true rate to boost recall

primary_test_scores = top_ens_test  # best ensemble by OOF F1
primary_thr = percentile_thr(primary_test_scores, 11)  # top 11% predicted anomalous

pred_dict = {int(u): float(s) for u, s in zip(test_users, primary_test_scores)}
np.savez("submission.npz", predictions=primary_test_scores)
with open("predictions.json", "w") as f:
    json.dump({"predictions": pred_dict}, f)

res_df = pd.DataFrame({
    "user": test_users, "anomaly_score": primary_test_scores,
    "predicted_label": (primary_test_scores >= primary_thr).astype(int)
}).sort_values("anomaly_score", ascending=False)
res_df.to_csv("predictions.csv", index=False)

n_anom = (primary_test_scores >= primary_thr).sum()
print("=" * 60)
print("PRIMARY SUBMISSION")
print("=" * 60)
print(f"Threshold: {primary_thr:.4f}")
print(f"Predicted anomalies: {n_anom}/{len(test_users)} ({n_anom/len(test_users)*100:.1f}%)")
print(f"Score stats: mean={primary_test_scores.mean():.4f}, std={primary_test_scores.std():.4f}")
print(f"\nTop 20 most anomalous users:")
print(res_df.head(20)[["user", "anomaly_score"]].to_string(index=False))

# Cross-check: what does OOF F1 look like at this percentile?
oof_thr_11 = percentile_thr(top_ens_oof, 11)
print(f"\nOOF sanity check at 11% threshold:")
print(f"  F1:        {f1_score(y, (top_ens_oof >= oof_thr_11).astype(int)):.4f}")
print(f"  Precision: {precision_score(y, (top_ens_oof >= oof_thr_11).astype(int)):.4f}")
print(f"  Recall:    {recall_score(y, (top_ens_oof >= oof_thr_11).astype(int)):.4f}")

PRIMARY SUBMISSION
Threshold: 0.1126
Predicted anomalies: 186/1625 (11.4%)
Score stats: mean=0.0628, std=0.1764

Top 20 most anomalous users:
 user  anomaly_score
 6384       1.000000
 5643       1.000000
 5565       1.000000
 6170       0.991525
 6535       0.991525
 5635       0.979898
 7005       0.979898
 5905       0.979898
 6207       0.979898
 6088       0.979898
 6660       0.979898
 6889       0.979898
 5724       0.979898
 6881       0.979898
 6036       0.979898
 6925       0.979898
 7102       0.979898
 6767       0.979898
 6685       0.935606
 6438       0.935606

OOF sanity check at 11% threshold:
  F1:        0.7873
  Precision: 0.6973
  Recall:    0.9038


## Summary: v3 → v4 Changes

### Root Cause Analysis
v3 achieved AUC=0.968 (great ranking) but F1=0.602 because:
- OOF threshold (0.4) was too conservative on test → only 4.5% predicted anomalous
- True anomaly rate ~8.5% → missed >50% of anomalies (recall=0.47)

### v4 Fixes
| Problem | v3 | v4 |
|---------|-----|-----|
| Threshold estimation | OOF F1-optimal | **Cross-batch validation + percentile-based** |
| Score calibration | None | **Isotonic regression on OOF scores** |
| Threshold bias | F1-symmetric | **F_beta=1.5 (recall-biased) option** |
| Predicted anomaly % | ~4.5% (too low) | **~10-12% (matched to expected rate)** |
| Unsupervised models | Features only | **Also direct ensemble members** |
| Submission strategy | Single best | **Multiple variants to test** |

### Expected Impact
- **Recall should increase from 0.47 → 0.70+** (predicting 2× more anomalies)
- **Precision may drop slightly from 0.83 → ~0.65-0.75**
- **Net F1 improvement from 0.602 → 0.70+** (better precision-recall balance)

### Which submission to try first?
1. **Primary (submission.npz)** — 11% percentile threshold (recommended)
2. **B_pctile10** — more conservative 10%
3. **C_fbeta_recall_biased** — if 11% is too aggressive
4. **D_cross_batch_thr** — data-driven threshold from cross-validation
